In [ ]:
cell = "K562"
resolution = 10000    # Hi-C resolution (bp): 10000(10kb), 25000(25kb)
display_reso = int(resolution / 1000)
root_dir = "/mnt/d/TAD-MultiOmicsNet"

In [ ]:

import pandas as pd
import numpy as np
import os

CURV_ROOT = f"{root_dir}/DNAcurve/norm/"
COORD_CSV = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/{cell}_coords_{display_reso}kb.csv"
OUT_DIR = f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/CURVE/"

signal_names = ["curv_mean"]
n_seg_total = 100  
seg_length = 1000

hic_df = pd.read_csv(COORD_CSV)
chrs_order = [f"chr{i}" for i in range(1, 23)]

os.makedirs(OUT_DIR, exist_ok=True)

for chrom in chrs_order:
    group_df = hic_df[hic_df["chr"] == chrom]
    print(f"Processing {chrom}, total bins: {len(group_df)}")
    curv_tsv_path = os.path.join(CURV_ROOT, f"{chrom}_curv1kb.tsv")
    if not os.path.exists(curv_tsv_path):
        print(f"!!! {curv_tsv_path} not found, skip {chrom}")
        continue
    curv_df = pd.read_csv(curv_tsv_path, sep="\t")
    
    curv_dict = {}
    for _, row in curv_df.iterrows():
        s = int(row["start"])
        feats = row[signal_names].values.astype(np.float32)
        curv_dict[s] = feats

    chr_features = []
    chr_masks = []

    for _, bin_row in group_df.iterrows():
        bin_start = int(bin_row["start"])
        bin_end = int(bin_row["end"])
        
        center = bin_start + (bin_end - bin_start) / 2
        win_left = int(center - 50000)
        raw_100 = np.full((n_seg_total, len(signal_names)), np.nan)
        for seg_idx in range(n_seg_total):
            seg_l = win_left + seg_idx * seg_length
            curv_start_candidate = int(np.floor(seg_l / seg_length) * seg_length)
            if curv_start_candidate in curv_dict:
                raw_100[seg_idx] = curv_dict[curv_start_candidate]
                
        out_mask = (~np.isnan(raw_100).all(axis=1)).astype(np.float32)
        out_feat = np.nan_to_num(raw_100, nan=0.0).astype(np.float32)
        chr_features.append(out_feat)
        chr_masks.append(out_mask)

    if len(chr_features) > 0:
        feat_arr = np.stack(chr_features, axis=0)
        mask_arr = np.stack(chr_masks, axis=0)
        out_npy = os.path.join(OUT_DIR, f"curv{cell}_{chrom}_predict_feat100.npy")
        out_mask = os.path.join(OUT_DIR, f"curv{cell}_{chrom}_predict_mask100.npy")
        np.save(out_npy, feat_arr)
        np.save(out_mask, mask_arr)
        print(f"--> {chrom} saved. feat shape: {feat_arr.shape}, mask shape: {mask_arr.shape}\n")
    else:
        print(f"--> {chrom} no valid bins, skip save\n")

print(f"\nFinished all chromosomes. hic csv total rows: {len(hic_df)}")
